# Matrices: Adding and Subtracting

Notebook 1 turned a system of equations into a single matrix equation. Before anything can
be done with that equation, the object on the left needs a proper introduction. A matrix is
a grid of numbers with a **shape**, an **address for every entry**, and a small set of
operations that behave exactly the way you would guess -- plus one rule about when they are
allowed at all.

This notebook covers the two friendliest operations, addition and subtraction, and the
scaling that goes with them. All three work entry by entry, all three keep the shape they
started with, and all three are defined only when the shapes agree exactly. NumPy usually
enforces that with a `ValueError` -- but not in every case, and the case where it stays
quiet is worth meeting early.

Everything here uses small whole numbers, so the arithmetic stays on paper -- which is where
it belongs while the ideas are new.

**Objectives:**
- Read the shape of a matrix and name any entry by its row and column
- Translate between the one-based entry $a_{ij}$ on paper and NumPy's zero-based `A[i-1, j-1]`
- Tell square from rectangular, and a row vector from a column vector
- Add and subtract matrices entry by entry, and state when that is defined
- Multiply a matrix by a single number
- Recognise the `ValueError` NumPy raises for most shape mismatches, and the broadcasting
  case where it raises nothing at all

**Reference:** See [`../GUIDE.md`](../GUIDE.md).

<!-- browser-runnable -->

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

## 1. A matrix is a grid of numbers

Plain English first. A **matrix** is a rectangular block of numbers written in rows and
columns. Nothing more. The numbers can be anything -- measurements, the coefficients of an
equation, the amplitudes of a quantum state -- and the grid is simply a way of keeping them
organised so that one symbol can stand for the whole block.

The one thing every matrix has is a **shape**: how many rows, and how many columns. Rows are
counted first, always, and the two counts are read with the word "by" between them. A grid
with 2 rows and 3 columns is a "2 by 3" matrix.

In NumPy you build one with `np.array` and a list of rows, and `.shape` hands the pair back
as a tuple -- in the same order.

In [ ]:
grid = np.array([
    [3, 1, 4],
    [1, 5, 9],
])
print(grid)
print("shape:", grid.shape)
print("rows:", grid.shape[0], "columns:", grid.shape[1])

In notation, a matrix with $m$ rows and $n$ columns is called an $m \times n$ matrix, and
the shape is written the same way round as `.shape` prints it: rows, then columns. The grid
above is $2 \times 3$.

## 2. Every entry has an address, and there is a trap in it

Plain English: to talk about one number inside the grid you say which row it is in and which
column. Row first, column second. Say that out loud a few times, because a swapped pair of
indices is the single most common hand-written matrix error there is.

Here is the trap, and it is better met now than in a debugging session. On paper, rows and
columns are numbered from **one**: the top-left entry sits in row 1, column 1. NumPy numbers
them from **zero**: the top-left entry is `grid[0, 0]`. The same entry therefore carries two
different addresses depending on which world you are in, and they differ by exactly one in
each slot.

In [ ]:
# The entry in row 2, column 3 -- one-based, the way it is written on paper.
# NumPy counts from zero, so the SAME entry is grid[1, 2].
print("row 2, column 3 =", grid[1, 2])
print("row 1, column 1 =", grid[0, 0])

# The trap, on purpose. grid[2, 3] is NOT "row 2, column 3": it asks for row
# index 2 and column index 3, and a 2-by-3 grid has neither.
try:
    print(grid[2, 3])
except IndexError as err:
    print("grid[2, 3] raised IndexError:", err)

In notation, the entry of a matrix $A$ in row $i$ and column $j$ is written $a_{ij}$:
lower-case letter, two subscripts, row first. So $a_{23}$ is the entry in row 2, column 3,
and in NumPy that is `A[1, 2]`. Translating either way, adjust each subscript by one.

## 3. Shapes have names

Plain English: a grid with as many rows as columns is **square**; anything else is
**rectangular**. Square matrices get most of the attention later, because they are the only
ones that can act on a vector and hand back something the same size -- which is exactly what
a quantum gate does.

Two rectangular shapes are common enough to have their own names:

- A **row vector** is a matrix with a single row: $1 \times n$.
- A **column vector** is a matrix with a single column: $n \times 1$.

Both are still matrices. The distinction matters because a $1 \times 4$ and a $4 \times 1$
hold the same four numbers and are *not* interchangeable -- as the next section's rule is
about to make blunt.

In [ ]:
square = np.array([[2, 7],
                   [0, 3]])          # 2 rows, 2 columns -- square
tall = np.array([[1],
                 [4],
                 [6]])               # 3 x 1 -- a column vector
wide = np.array([[5, 2, 8, 1]])      # 1 x 4 -- a row vector

for label, block in [("square", square), ("tall", tall), ("wide", wide)]:
    print(f"{label:7s} shape {block.shape}")

## 4. Two matrices that exist in every size, and what equality means

Plain English. The **zero matrix** is the grid whose every entry is 0. There is one for each
shape, and it does for matrices what 0 does for numbers: adding it changes nothing.

The **identity matrix** is square, carries 1s down the main diagonal -- top-left corner to
bottom-right -- and 0s everywhere else. It is named here only so the pattern is familiar
when it arrives for real: it earns its name in notebook 03, where multiplying by it leaves a
matrix untouched. For now, just learn to recognise it on sight.

**Equality** is stricter than it looks. Two matrices are equal only when they have the same
shape *and* every entry in every position agrees. A $2 \times 3$ is never equal to a
$3 \times 2$, whatever numbers are inside them.

In [ ]:
zeros23 = np.zeros((2, 3), dtype=int)
eye3 = np.eye(3, dtype=int)
print("the 2x3 zero matrix:")
print(zeros23)
print("the 3x3 identity:")
print(eye3)

# Equality means same shape AND every entry.
left = np.array([[1, 2], [3, 4]])
right = np.array([[1, 2], [3, 4]])
print("same shape, every entry agrees?", np.array_equal(left, right))
print("2x3 zero equals the 3x2 zero?", np.array_equal(zeros23, np.zeros((3, 2), dtype=int)))

In notation the zero matrix is written $0$ -- context supplies the shape -- and the
$n \times n$ identity is written $I_n$, or just $I$ when the size is obvious.

## 5. Adding and subtracting: entry by entry

Plain English, and this is the whole rule: to add two matrices, add the numbers sitting in
the same position. To subtract, subtract them. The answer has the same shape as the two you
started with, because every position of the answer came from a matching pair of positions.

The catch is that "the same position" has to mean something. If one grid has a slot the
other does not, there is no pair to add. So:

> **Two matrices can be added or subtracted only if they have exactly the same shape.**

Not "roughly the same size", and not "the same number of entries". A $2 \times 3$ and a
$3 \times 2$ both hold six numbers and still cannot be added, because no entry of one lines
up with an entry of the other.

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])
B = np.array([[10, 20, 30],
              [40, 50, 60]])

print("A + B =")
print(A + B)
print("A - B =")
print(A - B)
print("shape in, shape out:", A.shape, "and", B.shape, "give", (A + B).shape)

Now break it on purpose. `C` below is $2 \times 2$ while `A` is $2 \times 3$, so `A + C` has
no legal answer, and NumPy says so with a `ValueError` whose message names both shapes. A
$3 \times 2$ added to a $2 \times 3$ is refused the same way, even though the two grids hold
the same six numbers. Read those messages once here, while you know exactly what caused
them, and you will recognise them instantly the next time one surfaces in the middle of
something more complicated.

Then the uncomfortable half, and it matters more than the error does. **NumPy's `+` is not
matrix addition.** It is a wider operation called **broadcasting**: before adding, NumPy will
stretch any dimension of length 1 to match the other operand. Most mismatched shapes have no
such stretch available and raise, as the two pairs above do. But some mismatched shapes do,
and those add quite happily -- a $1 \times 3$ plus a $2 \times 1$ returns a $2 \times 3$ grid
that neither input had, with no error of any kind. A $1 \times 5$ plus a $5 \times 1$ returns
a $5 \times 5$.

That silent case is the dangerous one. An exception stops you; a wrong answer in a shape you
never asked for does not. So carry the mathematical rule, not "Python did not complain":
matrix addition is defined only for identical shapes, and the way to confirm you got matrix
addition is to look at the shape that came back.

In [ ]:
C = np.array([[1, 2],
              [3, 4]])       # 2x2 -- a different shape from A, which is 2x3

try:
    A + C
except ValueError as err:
    print("A + C raised ValueError:", err)

# Same six numbers on both sides, still nothing lines up.
try:
    np.zeros((3, 2), dtype=int) + np.zeros((2, 3), dtype=int)
except ValueError as err:
    print("3x2 + 2x3 raised ValueError:", err)

# The silent case: a row vector plus a column vector does NOT raise.
row = np.array([[1, 2, 3]])        # 1 x 3
col = np.array([[10],
                [20]])             # 2 x 1
stretched = row + col
print("1x3 + 2x1 gave shape", stretched.shape, "-- no error, and no matrix addition:")
print(stretched)

In notation, addition and subtraction are defined entry by entry,

$$(A + B)_{ij} = a_{ij} + b_{ij} \qquad (A - B)_{ij} = a_{ij} - b_{ij}$$

and both are defined only when $A$ and $B$ have the same shape. NumPy's `+` agrees with that
definition whenever the two shapes are identical, and goes beyond it otherwise.

## 6. Multiplying by a single number

Plain English: to scale a matrix, multiply **every** entry by the same number. Nothing is
skipped -- not the zeros, not the negatives -- and nothing moves position, so the shape
survives unchanged.

The number out front is called a **scalar**, to distinguish it from the matrix. Scaling is
what lets addition do more work than it appears to: $A + A$ is the same matrix as $2A$, and
subtracting is just adding a scaled-by-minus-one copy.

In [ ]:
D = np.array([[1, -2],
              [3,  0]])

print("3 times D =")
print(3 * D)
print("-1 times D =")
print(-1 * D)
print("D + D is the same matrix as 2 times D?", np.array_equal(D + D, 2 * D))
print("shape unchanged:", D.shape, "gives", (3 * D).shape)

In notation, scaling by a scalar $c$ is

$$(cA)_{ij} = c \, a_{ij}$$

and the scalar is written to the left of the matrix with no operator between them: $cA$,
$3A$, $-A$.

## 7. Notation cheat sheet

| Plain English | Notation | NumPy |
|---|---|---|
| A grid with $m$ rows and $n$ columns | $m \times n$ matrix | `A.shape` returns `(m, n)` |
| The entry in row $i$, column $j$ | $a_{ij}$, one-based | `A[i-1, j-1]`, zero-based |
| Grid of all zeros | $0$ | `np.zeros((m, n))` |
| 1s on the diagonal, 0s elsewhere | $I_n$ | `np.eye(n)` |
| Same shape, every entry agrees | $A = B$ | `np.array_equal(A, B)` |
| Add position by position | $(A + B)_{ij} = a_{ij} + b_{ij}$ | `A + B` |
| Subtract position by position | $(A - B)_{ij} = a_{ij} - b_{ij}$ | `A - B` |
| Multiply every entry by $c$ | $(cA)_{ij} = c \, a_{ij}$ | `c * A` |

Four habits worth carrying forward:

1. **Say the shape out loud before you operate.** Rows first. Most addition errors are shape
   errors that nobody checked.
2. **Adjust each subscript by one** when moving between paper and NumPy. The two worlds
   disagree by exactly that, in both slots.
3. **A `ValueError` about operands and shapes is a modelling message, not a NumPy quirk.** It
   means the two grids were never addable; go back and find which one is wrong. Its absence,
   though, proves nothing -- broadcasting turns some mismatches into a silently bigger grid,
   so check the shape that came back.
4. **Do the small ones on paper.** Entry-wise arithmetic is where the intuition is built, and
   the window closes as soon as the grids get big.

## 8. Drills: practise until it is dull

Reading matrix arithmetic and being able to do it are different skills, and only one of them
survives contact with the next notebook. `drill` generates an endless supply of small
problems and marks them without handing anything over: `check` reports which entries are off
and by how much, and `reveal` is the only door to the worked answer.

Three kinds are in scope after this notebook -- `"add"`, `"subtract"` and `"scale"`. Level 1
is small non-negative 2x2 grids, level 2 brings in negatives and 3x3, and level 3 goes
rectangular. Pass a `seed` for a reproducible problem, or leave it out and the module draws
one and prints it, so you can come back to the same drill later.

The cell below only poses the problem. Work it on paper, then uncomment the `check` line and
type your two rows into it. Change the kind, the level and the seed and re-run as often as
you like. Once the drills feel dull, the eight exercises below will feel easy.

In [ ]:
from lib.linalg_drills import drill

d = drill("add", level=1, seed=7)
d.show()

# Work it on paper. Then replace the ellipses with your own two rows,
# uncomment the line, and re-run this cell:
# d.check([[..., ...], [..., ...]])

# Still stuck? reveal() is the only path to the worked answer -- use it last.
# d.reveal()

### Exercise 1 — Read a matrix

Here is a matrix, and nothing about it needs computing -- only reading.

$$P = \begin{pmatrix} 4 & 0 & 7 \\ 2 & 9 & 1 \end{pmatrix}$$

Define `ex1_shape` as the shape of $P$: a tuple, rows first. Define `ex1_p13` as the value of
the entry $p_{13}$, and `ex1_p21` as the value of the entry $p_{21}$. Both subscripts are the
one-based, on-paper kind from Section 2, so read the values straight off the grid above as
plain integers -- no NumPy required for any of this.

<details><summary>Hint 1 — nudge</summary>

Two counts make a shape, and the order they are spoken in is fixed. For the entries,
remember which of the two subscripts names the row: Section 2 says it is always the same
one, and getting it backwards is the most common slip in this whole notebook.

</details>
<details><summary>Hint 2 — approach</summary>

Count the horizontal lines of numbers, then the vertical ones, and write the pair as a tuple
in that order. For each entry, walk down to the row the first subscript names, then across
to the column the second names, and read off what is sitting there. Three plain assignments:
one tuple and two integers.

</details>

In [ ]:
# Exercise 1: Read the shape and two named entries off the matrix P.
# Define: ex1_shape, ex1_p13, ex1_p21

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    e1_p = np.array([[4, 0, 7], [2, 9, 1]])
    assert len(tuple(ex1_shape)) == 2, (
        "a shape is a pair of counts -- how many rows, then how many columns"
    )
    assert tuple(ex1_shape) == e1_p.shape, (
        "count the horizontal lines of numbers first and the vertical ones second, "
        "and report them in that order"
    )
    assert int(ex1_p13) == int(e1_p[0, 2]), (
        "the first subscript of p_13 names the row and the second names the column -- "
        "walk down first, then across"
    )
    assert int(ex1_p21) == int(e1_p[1, 0]), (
        "p_21 sits in row 2, column 1 and holds a different number from p_12 in row 1, "
        "column 2 -- re-read which subscript comes first"
    )

### Exercise 2 — Add two 2x2 matrices by hand

$$G = \begin{pmatrix} 2 & 5 \\ 7 & 1 \end{pmatrix} \qquad H = \begin{pmatrix} 3 & 4 \\ 6 & 8 \end{pmatrix}$$

Work out $G + H$ on paper, then define `ex2_sum` as that matrix -- a nested list or a NumPy
array, either grades. Resist letting NumPy add them for you; the four additions are the
whole point of this one.

<details><summary>Hint 1 — nudge</summary>

Addition pairs up the entries that sit in the same position, so the answer has exactly one
entry per position of the originals -- and therefore exactly the same shape.

</details>
<details><summary>Hint 2 — approach</summary>

Go position by position: top-left with top-left, top-right with top-right, and so on through
all four. Write the four results into the same two-by-two arrangement, then type that
arrangement in as a nested list, one inner list per row.

</details>

In [ ]:
# Exercise 2: Compute G + H by hand.
# Define: ex2_sum

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    e2_g = np.array([[2, 5], [7, 1]])
    e2_h = np.array([[3, 4], [6, 8]])
    assert np.asarray(ex2_sum).shape == (2, 2), (
        "addition never changes the shape -- two 2x2 grids give a 2x2 answer, written as "
        "two rows of two"
    )
    assert np.allclose(np.asarray(ex2_sum), e2_g + e2_h), (
        "each entry of the answer is the sum of the two entries in that same position -- "
        "the bottom-left pair is the only one whose total passes ten"
    )

### Exercise 3 — Add two 2x3 matrices by hand

Same operation, a wider grid, and negative numbers in the mix.

$$R = \begin{pmatrix} 1 & -2 & 4 \\ 0 & 6 & -3 \end{pmatrix} \qquad S = \begin{pmatrix} 5 & 3 & -1 \\ 7 & -2 & 8 \end{pmatrix}$$

Define `ex3_sum` as $R + S$, worked on paper. Six additions, and four of them involve a
negative number.

<details><summary>Hint 1 — nudge</summary>

Nothing about the rule changes when the grid gets wider or the numbers go negative. Only the
count of pairs changes -- and adding a negative number is the same move as subtracting its
size.

</details>
<details><summary>Hint 2 — approach</summary>

Take the top row of both matrices and add across it, giving three results; then the bottom
row, giving three more. Keep each sign attached to the number it belongs to as you go, and
assemble the six results into two rows of three.

</details>

In [ ]:
# Exercise 3: Compute R + S by hand.
# Define: ex3_sum

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    e3_r = np.array([[1, -2, 4], [0, 6, -3]])
    e3_s = np.array([[5, 3, -1], [7, -2, 8]])
    assert np.asarray(ex3_sum).shape == (2, 3), (
        "the sum keeps the shape of what went in -- 2 rows of 3, not 3 rows of 2"
    )
    assert np.allclose(np.asarray(ex3_sum), e3_r + e3_s), (
        "look again at the four positions where a negative meets a positive -- adding a "
        "negative moves the total down, not up"
    )

### Exercise 4 — Subtract two 3x3 matrices by hand

$$T = \begin{pmatrix} 9 & 4 & 1 \\ 0 & 7 & 5 \\ 3 & 2 & 8 \end{pmatrix} \qquad U = \begin{pmatrix} 2 & 6 & 3 \\ 4 & 1 & 5 \\ 7 & 5 & 9 \end{pmatrix}$$

Define `ex4_diff` as $T - U$, in that order, worked on paper. Nine subtractions, and several
of them land below zero.

<details><summary>Hint 1 — nudge</summary>

Subtraction is entry-wise for the same reason addition is, and it obeys the same shape rule.
What it does not obey is symmetry: swapping the two matrices does not give the same answer,
so the order stated in the prompt is part of the question.

</details>
<details><summary>Hint 2 — approach</summary>

For each of the nine positions, take the entry from the first matrix and remove the entry
from the second. Where the second is the bigger of the two, the result is negative -- write
that minus sign down as you go rather than tidying it up afterwards.

</details>

In [ ]:
# Exercise 4: Compute T - U by hand.
# Define: ex4_diff

# TODO: your code here

In [ ]:
# Check Exercise 4 -- run after your attempt.
from lib.grading import check

with check("Exercise 4"):
    e4_t = np.array([[9, 4, 1], [0, 7, 5], [3, 2, 8]])
    e4_u = np.array([[2, 6, 3], [4, 1, 5], [7, 5, 9]])
    assert np.asarray(ex4_diff).shape == (3, 3), (
        "subtracting two 3x3 grids leaves a 3x3 grid -- three rows of three"
    )
    assert not np.allclose(np.asarray(ex4_diff), e4_u - e4_t), (
        "every sign is flipped, which means the two matrices went in the other way round "
        "-- the prompt asks for the second to be removed from the first"
    )
    assert np.allclose(np.asarray(ex4_diff), e4_t - e4_u), (
        "go position by position: six of the nine results are negative, one lands exactly "
        "on 0, and a smaller number minus a larger one is always negative"
    )

### Exercise 5 — Scale a matrix by hand

$$V = \begin{pmatrix} 2 & -1 \\ 0 & 4 \\ 3 & 5 \end{pmatrix}$$

Define `ex5_scaled` as $3V$, worked on paper. Note the shape of $V$ before you start: it is
rectangular, and the scalar does not care.

<details><summary>Hint 1 — nudge</summary>

A scalar reaches every single entry. Nothing is exempt -- not the negative one, and not the
zero -- and no entry moves position, so the shape comes through intact.

</details>
<details><summary>Hint 2 — approach</summary>

Walk the grid one entry at a time, multiply each by the scalar, and write the result into the
slot it came from. Six entries in, six entries out, in the same three-rows-of-two
arrangement.

</details>

In [ ]:
# Exercise 5: Compute 3V by hand.
# Define: ex5_scaled

# TODO: your code here

In [ ]:
# Check Exercise 5 -- run after your attempt.
from lib.grading import check

with check("Exercise 5"):
    e5_v = np.array([[2, -1], [0, 4], [3, 5]])
    assert np.asarray(ex5_scaled).shape == (3, 2), (
        "scaling moves nothing, so the answer keeps V's own 3-rows-by-2-columns shape"
    )
    assert np.allclose(np.asarray(ex5_scaled), 3 * e5_v), (
        "the scalar has to reach every entry -- check the one negative entry and the one "
        "zero, which are the two most often left alone"
    )

### Exercise 6 — Which pairs can be added?

Five pairs of matrices are described below by their shapes alone. For each pair, decide
whether **matrix addition** of the two is defined -- the rule from Section 5, not whatever
NumPy's `+` might broadcast its way through.

1. a $2 \times 3$ and a $2 \times 3$
2. a $3 \times 2$ and a $2 \times 3$
3. a $4 \times 4$ and a $4 \times 4$
4. a $1 \times 5$ and a $5 \times 1$
5. a $2 \times 2$ and a $2 \times 3$

Define `ex6_can_add` as a list of five values, each `True` or `False`, in the order the pairs
are listed above.

<details><summary>Hint 1 — nudge</summary>

Section 5 states the rule in a single sentence, and it is not about how many numbers each
grid holds. Two of these pairs put the same count of numbers on each side and still fail it,
and one of those two is the pair NumPy would broadcast rather than reject -- broadcasting is
not addition.

</details>
<details><summary>Hint 2 — approach</summary>

Compare the two shapes of each pair slot by slot: first count against first count, second
against second. The sum is defined only when both comparisons agree. Write the five verdicts
into a list, keeping the order of the pairs above.

</details>

In [ ]:
# Exercise 6: Decide which of the five shape pairs can be added.
# Define: ex6_can_add

# TODO: your code here

In [ ]:
# Check Exercise 6 -- run after your attempt.
from lib.grading import check

with check("Exercise 6"):
    e6_pairs = [
        ((2, 3), (2, 3)),
        ((3, 2), (2, 3)),
        ((4, 4), (4, 4)),
        ((1, 5), (5, 1)),
        ((2, 2), (2, 3)),
    ]
    assert len(ex6_can_add) == 5, (
        "one verdict per pair, five in all, kept in the order the pairs are listed"
    )
    assert [bool(x) for x in ex6_can_add] == [a == b for a, b in e6_pairs], (
        "both counts of the shape have to agree, not just one of them and not the total "
        "number of entries -- pair 5 agrees on rows only, and pair 4 is the one NumPy "
        "would broadcast into a 5x5 instead of refusing"
    )

### Exercise 7 — Build a zero matrix and an identity

Define `ex7_zero` as the $3 \times 4$ zero matrix, and `ex7_identity` as the $4 \times 4$
identity matrix. Section 4 describes both in words; build them here as NumPy arrays -- or as
nested lists, which grade the same.

Watch the two shapes. One is rectangular and one is square, and only one of them could ever
be an identity.

<details><summary>Hint 1 — nudge</summary>

The zero matrix is defined by what every entry is. The identity is defined by where the 1s
sit and what fills the rest. Neither definition needs a formula -- either can be typed out
by hand if you would rather see it than call something.

</details>
<details><summary>Hint 2 — approach</summary>

NumPy has a constructor for each. One takes a shape tuple and fills it with a single value;
the other takes a single size and places 1s down the main diagonal. Both appeared in Section
4, at different sizes from the ones asked for here -- change the arguments, not the
functions.

</details>

In [ ]:
# Exercise 7: Build the 3x4 zero matrix and the 4x4 identity.
# Define: ex7_zero, ex7_identity

# TODO: your code here

In [ ]:
# Check Exercise 7 -- run after your attempt.
from lib.grading import check

with check("Exercise 7"):
    assert np.asarray(ex7_zero).shape == (3, 4), (
        "3 by 4 means 3 rows and 4 columns, in that order -- the two are not interchangeable"
    )
    assert np.asarray(ex7_identity).shape == (4, 4), (
        "an identity is square, and this one is asked for at size 4"
    )
    assert np.allclose(np.asarray(ex7_zero), np.zeros((3, 4))), (
        "every single entry of a zero matrix is 0"
    )
    e7_diag = np.array([[1 if r == c else 0 for c in range(4)] for r in range(4)])
    assert np.allclose(np.asarray(ex7_identity), e7_diag), (
        "1s run down the main diagonal, from the top-left corner to the bottom-right, "
        "and every other position holds 0"
    )

### Exercise 8 — Let NumPy do it, then check your paper

The closer puts all three operations into one expression. Three fresh grids, with names of
their own so nothing the teaching cells above left lying around can be mistaken for them:

$$J = \begin{pmatrix} 5 & 1 \\ 2 & 8 \end{pmatrix} \qquad K = \begin{pmatrix} 3 & 6 \\ 7 & 0 \end{pmatrix} \qquad L = \begin{pmatrix} 1 & 2 \\ 4 & 3 \end{pmatrix}$$

Build the three matrices as NumPy arrays named `ex8_j`, `ex8_k` and `ex8_l`. Define
`ex8_result` as $J + K - 2L$, computed by NumPy in a single expression. Then work the same
expression on paper and define `ex8_by_hand` as what you got. The check passes only when
both agree with the arithmetic -- which is the entire reason for doing it twice.

<details><summary>Hint 1 — nudge</summary>

The scalar attaches to the matrix immediately after it, and that scaling happens before the
subtraction does -- exactly the precedence you already expect from ordinary numbers.

</details>
<details><summary>Hint 2 — approach</summary>

Type the three grids in as nested lists, one inner list per row. Then write the expression
with the same `+`, `-` and `*` you would use on plain numbers; NumPy applies each of them
entry by entry. On paper, scale the third grid first, then run the two entry-wise steps left
to right.

</details>

In [ ]:
# Exercise 8: Compute J + K - 2L with NumPy, and again on paper.
# Define: ex8_j, ex8_k, ex8_l, ex8_result, ex8_by_hand

# TODO: your code here

In [ ]:
# Check Exercise 8 -- run after your attempt.
from lib.grading import check

with check("Exercise 8"):
    e8_expected = (
        np.array([[5, 1], [2, 8]])
        + np.array([[3, 6], [7, 0]])
        - 2 * np.array([[1, 2], [4, 3]])
    )
    assert np.asarray(ex8_j).shape == (2, 2), "every grid in the prompt is two rows of two"
    assert np.asarray(ex8_k).shape == (2, 2), "every grid in the prompt is two rows of two"
    assert np.asarray(ex8_l).shape == (2, 2), "every grid in the prompt is two rows of two"
    assert np.allclose(np.asarray(ex8_j), [[5, 1], [2, 8]]), (
        "re-read the grid named J in the prompt, one row at a time"
    )
    assert np.allclose(np.asarray(ex8_k), [[3, 6], [7, 0]]), (
        "re-read the grid named K in the prompt, one row at a time"
    )
    assert np.allclose(np.asarray(ex8_l), [[1, 2], [4, 3]]), (
        "re-read the grid named L in the prompt, one row at a time"
    )
    assert np.asarray(ex8_result).shape == (2, 2), (
        "adding and scaling both keep the shape, so the result is 2x2 as well"
    )
    assert np.allclose(np.asarray(ex8_result), e8_expected), (
        "let NumPy do the arithmetic -- combine the three arrays and the scalar in one "
        "expression rather than typing an answer in"
    )
    assert np.allclose(np.asarray(ex8_by_hand), e8_expected), (
        "your paper answer and NumPy's disagree somewhere -- redo the position where they "
        "differ, scaling L by 2 before you subtract it"
    )

### Solutions

Worked answers to all eight. Run your own attempt and its check first -- a check tells you
where you are without telling you what to write, and that gap is where the learning happens.

In [ ]:
# --- Exercise 1 ---
ex1_shape = (2, 3)                       # 2 rows, 3 columns -- rows first, always
ex1_p13 = 7                              # row 1, column 3
ex1_p21 = 2                              # row 2, column 1 -- a different entry from p_12
print("shape:", ex1_shape, " p_13 =", ex1_p13, " p_21 =", ex1_p21)

# --- Exercise 2 ---
ex2_sum = np.array([[2 + 3, 5 + 4],      # position by position
                    [7 + 6, 1 + 8]])
print("G + H =")
print(ex2_sum)

# --- Exercise 3 ---
ex3_sum = np.array([[1 + 5, -2 + 3, 4 + -1],
                    [0 + 7, 6 + -2, -3 + 8]])
print("R + S =")
print(ex3_sum)

# --- Exercise 4 ---
ex4_diff = np.array([[9 - 2, 4 - 6, 1 - 3],
                     [0 - 4, 7 - 1, 5 - 5],
                     [3 - 7, 2 - 5, 8 - 9]])
print("T - U =")
print(ex4_diff)

# --- Exercise 5 ---
ex5_scaled = np.array([[3 * 2, 3 * -1],  # the scalar reaches every entry
                       [3 * 0, 3 * 4],
                       [3 * 3, 3 * 5]])
print("3V =")
print(ex5_scaled)

# --- Exercise 6 ---
# Addable exactly when BOTH counts of the shape agree. Pair 4 is the trap:
# NumPy broadcasts a 1x5 and a 5x1 into a 5x5, but that is not matrix addition.
ex6_can_add = [True, False, True, False, False]
print(ex6_can_add)

# --- Exercise 7 ---
ex7_zero = np.zeros((3, 4), dtype=int)   # a shape tuple, rows first
ex7_identity = np.eye(4, dtype=int)      # 1s down the main diagonal
print(ex7_zero)
print(ex7_identity)

# --- Exercise 8 ---
ex8_j = np.array([[5, 1], [2, 8]])
ex8_k = np.array([[3, 6], [7, 0]])
ex8_l = np.array([[1, 2], [4, 3]])
ex8_result = ex8_j + ex8_k - 2 * ex8_l
ex8_by_hand = np.array([[5 + 3 - 2, 1 + 6 - 4],
                        [2 + 7 - 8, 8 + 0 - 6]])
print("NumPy:", ex8_result.tolist(), " by hand:", ex8_by_hand.tolist())

## 9. Where this shows up: a qubit is a column

Everything above was arithmetic on grids of whole numbers. Here is the payoff, and it is much
closer than it looks.

A single qubit's state is a **column vector with two entries** -- a $2 \times 1$ matrix. The
two basis states are exactly the two columns you would guess:

$$\lvert 0 \rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix} \qquad \lvert 1 \rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$

The famous superposition, the one written $\lvert + \rangle$, is built out of those two using
only the operations this notebook taught:

$$\lvert + \rangle = \tfrac{1}{\sqrt{2}} \left( \begin{pmatrix} 1 \\ 0 \end{pmatrix} + \begin{pmatrix} 0 \\ 1 \end{pmatrix} \right)$$

Read that from the inside out. The plus sign between the brackets is **entry-wise matrix
addition** -- the same operation you performed four times in Exercise 2, and six more times in
Exercise 3. The fraction out front is **scalar multiplication** -- the same operation as
Exercise 5, with a scalar that happens not to be a whole number. Two $2 \times 1$ columns, one
addition, one scaling, and the result is the state that makes a quantum computer interesting.

That is the honest answer to "why start with grids of integers". Superposition is not a new
kind of arithmetic; it is this arithmetic, applied to columns of amplitudes. What is still
missing is the rule deciding which columns count as legal states -- amplitudes are allowed to
be complex numbers, and their squared sizes have to add to one. Both arrive in `00-prereqs`,
and by then the addition underneath will be automatic.

## Summary

- A **matrix** is a grid of numbers, and its **shape** is the pair (rows, columns) -- rows
  first, every time, in speech and in `.shape`.
- Entries are addressed row-then-column. On paper $a_{ij}$ is **one-based**; in NumPy the same
  entry is `A[i-1, j-1]`, **zero-based**. Adjust each subscript by one when you cross between
  the two.
- **Square** means as many rows as columns. A **row vector** is $1 \times n$, a **column
  vector** is $n \times 1$, and the two are not interchangeable.
- The **zero matrix** exists in every shape and adds nothing. The **identity** is square, with
  1s down the main diagonal; it earns its name in notebook 03.
- **Equality** needs the same shape *and* every matching entry to agree.
- **Addition and subtraction are entry-wise**, they preserve the shape, and they are defined
  **only for identical shapes**. Most mismatches give a `ValueError` naming both shapes -- that
  message is a modelling error surfacing early, not a NumPy inconvenience.
- **NumPy's `+` is broadcasting, not matrix addition.** When a dimension of length 1 can be
  stretched to fit -- a $1 \times 3$ plus a $2 \times 1$, say -- no error comes at all and you
  get back a grid neither input had. Check the shape of the result; silence is not agreement.
- **Scaling** multiplies every entry by one number and leaves the shape alone. $A + A$ is the
  same matrix as $2A$.

**You finished notebook 2.** Next is
[`03-matrix-multiplication.ipynb`](03-matrix-multiplication.ipynb), where the first operation
that is *not* entry-wise arrives -- and the shape rule stops being about equality.